# Task 13: Temporal Convolutional Network (TCN) with Dilated Causal Convolutions

## Objective

Implement a Temporal Convolutional Network for multi-dimensional time-series prediction.

The architecture uses:

- Causal 1D convolutions
- Dilated convolutions
- Residual connections
- Multiple dilation factors
- Historical-window prediction

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)

device = "cuda" if torch.cuda.is_available() else "cpu"


# ============================================================
# Synthetic multi-dimensional time series
# ============================================================

T = 3000
t = np.arange(T)

series = np.stack([
    np.sin(t / 25),
    np.cos(t / 40),
    np.sin(t / 70) + 0.3 * np.sin(t / 10)
], axis=1)

series += np.random.normal(
    0,
    0.05,
    series.shape
)

series = torch.tensor(
    series,
    dtype=torch.float32
)

WINDOW = 64

X = []
Y = []

for i in range(
    T - WINDOW - 1
):

    X.append(
        series[i:i + WINDOW]
    )

    Y.append(
        series[i + WINDOW]
    )

X = torch.stack(X)
Y = torch.stack(Y)

split = int(
    0.8 * len(X)
)

X_train = X[:split]
Y_train = Y[:split]

X_test = X[split:]
Y_test = Y[split:]

print("Training:", X_train.shape)
print("Testing :", X_test.shape)

# Dilated Causal Convolution

For a causal convolution, padding is applied only to historical positions.

The dilation determines how far apart convolutional kernel elements are:

\[
y_t =
\sum_k w_k x_{t-kd}
\]

Therefore the model never accesses future observations.

Residual connections help preserve information through deeper temporal layers.

In [ ]:
class CausalConv1D(nn.Module):

    def __init__(
        self,
        in_channels,
        out_channels,
        kernel_size,
        dilation
    ):
        super().__init__()

        self.padding = (
            kernel_size - 1
        ) * dilation

        self.conv = nn.Conv1d(
            in_channels,
            out_channels,
            kernel_size,
            dilation=dilation,
            padding=self.padding
        )

    def forward(self, x):

        y = self.conv(x)

        # Remove future-side padding
        if self.padding > 0:
            y = y[:, :, :-self.padding]

        return y


class TCNBlock(nn.Module):

    def __init__(
        self,
        channels,
        dilation
    ):
        super().__init__()

        self.conv1 = CausalConv1D(
            channels,
            channels,
            3,
            dilation
        )

        self.conv2 = CausalConv1D(
            channels,
            channels,
            3,
            dilation
        )

        self.relu = nn.ReLU()

    def forward(self, x):

        residual = x

        y = self.relu(
            self.conv1(x)
        )

        y = self.relu(
            self.conv2(y)
        )

        return y + residual


class TCN(nn.Module):

    def __init__(
        self,
        input_dim=3,
        channels=32
    ):
        super().__init__()

        self.input = nn.Conv1d(
            input_dim,
            channels,
            1
        )

        self.blocks = nn.Sequential(
            TCNBlock(channels, 1),
            TCNBlock(channels, 2),
            TCNBlock(channels, 4),
            TCNBlock(channels, 8),
            TCNBlock(channels, 16)
        )

        self.output = nn.Linear(
            channels,
            input_dim
        )

    def forward(self, x):

        # B,T,C -> B,C,T
        x = x.transpose(1, 2)

        x = self.input(x)

        x = self.blocks(x)

        # Last historical timestep
        x = x[:, :, -1]

        return self.output(x)


model = TCN().to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3
)

loss_fn = nn.MSELoss()

In [ ]:
# ============================================================
# Train TCN
# ============================================================

X_train_device = X_train.to(device)
Y_train_device = Y_train.to(device)

history = []

for epoch in range(20):

    model.train()

    optimizer.zero_grad()

    predictions = model(
        X_train_device
    )

    loss = loss_fn(
        predictions,
        Y_train_device
    )

    loss.backward()
    optimizer.step()

    history.append(
        loss.item()
    )

    if (epoch + 1) % 5 == 0:

        print(
            f"Epoch {epoch+1}/20 - "
            f"Loss: {loss.item():.6f}"
        )


# ============================================================
# Evaluation
# ============================================================

model.eval()

with torch.no_grad():

    predictions = model(
        X_test.to(device)
    ).cpu()

test_mse = loss_fn(
    predictions,
    Y_test
).item()

print("Test MSE:", test_mse)


plt.figure(figsize=(7, 4))
plt.plot(history)
plt.xlabel("Epoch")
plt.ylabel("MSE")
plt.title("TCN Training Convergence")
plt.grid(True)
plt.show()

# Conclusion

A Temporal Convolutional Network was implemented using causal and dilated 1D convolutions.

Dilation factors of:

\[
1,2,4,8,16
\]

allowed the network to capture both short-term and long-range temporal dependencies while preserving causal ordering.

Residual connections improved information flow through the temporal network. The final model was evaluated using mean squared error on unseen future sequences.